# BudgetBot - A Chatbot to understand Indian financial budget 2024 (Announced July 2024)

In [ ]:
# Install dependencies for local Python 3.10 and Colab


In [ ]:
import requests
import openai
import os


In [ ]:
# Set OPENAI_API_KEY in environment variables, do not hardcode it here


### Download data

In [ ]:
from pathlib import Path


def download_data(url, output_path="budget_2024.pdf"):
    output_path = Path(output_path)
    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"Using existing file: {output_path.resolve()}")
        return output_path

    response = requests.get(url, timeout=60)
    response.raise_for_status()
    output_path.write_bytes(response.content)
    print(f"Downloaded {len(response.content):,} bytes to {output_path.resolve()}")
    return output_path


In [ ]:
url = "https://www.indiabudget.gov.in/doc/budget_speech.pdf"
download_data(url)


In [ ]:
from pathlib import Path

for path in sorted(Path('.').glob('*')):
    print(path)


### RAG

In [ ]:
# Import necessary classes from the llama_index package
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader


In [ ]:
# Read documents from the specified directory and load a specific document.
documents = SimpleDirectoryReader("./").load_data("budget_2024.pdf")


In [ ]:
documents[0:5]


### Vector DB

In [ ]:
from llama_index.core.vector_stores.simple import SimpleVectorStore
from llama_index.core import StorageContext


In [ ]:
vector_store = SimpleVectorStore()
storage_context = StorageContext.from_defaults(vector_store=vector_store)


In [ ]:
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)
query_engine = index.as_query_engine()


In [ ]:
# query="What is the outlook for Indian GDP in 2024?"
query = "highlight the indian budget 2024 in 10 bullet points"


In [ ]:

response = query_engine.query(query)
print(response)


### Pass the Query and Context to LLM

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=openai.api_key)


In [ ]:
def call_gpt_3_5_turbo(question, answers):
    """Call GPT-3.5 Turbo API and get the best possible answer."""
    prompt = f"Based on the following question and the top answer, provide the best possible answer. If none of the answers are satisfactory, generate a new answer based on your own knowledge.\n\nQuestion: {question}\n"
    prompt += answers
    # prompt += "\nPlease provide the best answer or state that the answer is not present in the given answers and generate a new one if necessary."
    
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    result = response.choices[0].message.content
    return result


In [ ]:
# Get the final answer from GPT-3.5 Turbo
final_answer = call_gpt_3_5_turbo(query, str(response))
print(final_answer)
